# Question 2 — Time Series Analysis: Air Passengers


## Part 1

**Autocorrelation Function (ACF)** measures the correlation between a time series $y_t$ and its own past values $y_{t-k}$ for a given lag $k$, without removing the influence of intermediate lags. Formally,
$$\rho_k = \frac{\text{Cov}(y_t, y_{t-k})}{\text{Var}(y_t)}.$$
Because it includes both direct and indirect dependencies, the ACF of a stationary AR($p$) process decays slowly (geometrically or in a damped sinusoidal pattern) rather than cutting off sharply.

**Partial Autocorrelation Function (PACF)** measures the correlation between $y_t$ and $y_{t-k}$ after controlling for all shorter lags $y_{t-1}, \ldots, y_{t-k+1}$. It isolates the direct linear relationship at lag $k$. For a pure AR($p$) process, the PACF cuts off to zero after lag $p$, making it the primary diagnostic for identifying the AR order.

## Part 2

Given a stationary time series, the workflow is:

1. **Check stationarity** — inspect the raw series and apply differencing or log-transformation if a trend or heteroscedasticity is present. Confirm with the Augmented Dickey-Fuller (ADF) test.
2. **Plot the PACF** — count the number of lags at which the partial autocorrelation is statistically significant. That count is your candidate AR order $p$.
3. **Plot the ACF** — a slowly decaying ACF is consistent with an AR process; a sharp cutoff at some lag $q$ instead suggests an MA component.
4. **Cross-validate using information criteria** — fit AR models for a range of $p$ values and compare AIC/BIC scores. The model with the lowest information criterion is preferred.
5. **Diagnose residuals** — the residuals of the chosen model should approximate white noise (ACF of residuals should be within confidence bands at all lags).

## Part 3 — Air Passengers Dataset

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from statsmodels.graphics.tsaplots import acf, pacf
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_squared_error

pio.renderers.default = 'notebook'

In [ ]:
df = pd.read_csv('AirPassengers.xls')  # file is CSV despite .xls extension
df.columns = ['Month', 'Passengers']
df['Month'] = pd.to_datetime(df['Month'])
df = df.set_index('Month').sort_index()
df.head()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df.index, y=df['Passengers'],
    mode='lines+markers', name='Passengers',
    line=dict(color='steelblue')
))
fig.update_layout(
    title='Monthly Air Passengers (1949–1960)',
    xaxis_title='Month', yaxis_title='Number of Passengers',
    template='plotly_white'
)
fig.show()

In [ ]:
log_passengers = np.log(df['Passengers'])

# First-order difference of log series to remove trend
log_diff = log_passengers.diff().dropna()

adf_stat, adf_p, _, _, crit, _ = adfuller(log_diff)
print(f"ADF statistic: {adf_stat:.4f}, p-value: {adf_p:.4f}")
print("Series is stationary (p < 0.05)" if adf_p < 0.05 else "Series may not be stationary")

In [ ]:
nlags = 30
acf_vals, acf_ci  = acf(log_diff,  nlags=nlags, alpha=0.05)
pacf_vals, pacf_ci = pacf(log_diff, nlags=nlags, alpha=0.05)

lags = np.arange(len(acf_vals))
conf = 1.96 / np.sqrt(len(log_diff))

def make_correlogram(vals, ci, title):
    fig = go.Figure()
    # Stem bars
    for i, v in enumerate(vals):
        fig.add_shape(type='line',
            x0=i, x1=i, y0=0, y1=v,
            line=dict(color='steelblue', width=2)
        )
    fig.add_trace(go.Scatter(
        x=lags, y=vals, mode='markers',
        marker=dict(color='steelblue', size=6), showlegend=False
    ))
    # Confidence bands
    fig.add_hline(y=conf,  line_color='red', line_dash='dash', annotation_text='+95% CI')
    fig.add_hline(y=-conf, line_color='red', line_dash='dash', annotation_text='-95% CI')
    fig.add_hline(y=0, line_color='black', line_width=0.5)
    fig.update_layout(
        title=title, xaxis_title='Lag', yaxis_title='Correlation',
        template='plotly_white', yaxis=dict(range=[-1, 1])
    )
    return fig

make_correlogram(acf_vals,  acf_ci,  'ACF — Log-Differenced Air Passengers').show()
make_correlogram(pacf_vals, pacf_ci, 'PACF — Log-Differenced Air Passengers').show()

In [ ]:
# Train: 1949-01 to 1959-12  |  Test: 1960-01 to 1960-12
train_log = log_passengers['1949':'1959']
test_log  = log_passengers['1960']

# Differenced train series for model fitting
train_diff = train_log.diff().dropna()

print(f"Train: {len(train_log)} months, Test: {len(test_log)} months")

In [ ]:
# Fit AR(13) on log-differenced train series
ar_order = 13
model = AutoReg(train_diff, lags=ar_order, old_names=False)
result = model.fit()
print(result.summary())

In [ ]:
# One-step-ahead rolling forecast on the differenced test series
# We append actual values progressively (walk-forward validation)
history = list(train_diff.values)
predictions_diff = []

test_diff_actual = test_log.diff().dropna()  # 11 values (Jan 1960 diff is NaN)
# For the first test step we use the last train log value
test_log_actual = test_log.values

# We forecast the log-differenced series one step at a time
for i in range(len(test_log)):
    train_subset = np.array(history)
    m = AutoReg(train_subset, lags=ar_order, old_names=False).fit()
    pred = m.predict(start=len(train_subset), end=len(train_subset))[0]
    predictions_diff.append(pred)
    # Append actual differenced value for next step
    if i == 0:
        # first diff value: log(test[0]) - log(train[-1])
        actual_diff_val = test_log_actual[0] - train_log.values[-1]
    else:
        actual_diff_val = test_log_actual[i] - test_log_actual[i-1]
    history.append(actual_diff_val)

# Convert differenced log predictions back to original scale
log_pred = np.zeros(len(test_log))
log_pred[0] = train_log.values[-1] + predictions_diff[0]
for i in range(1, len(test_log)):
    log_pred[i] = test_log_actual[i-1] + predictions_diff[i]

pred_passengers = np.exp(log_pred)
actual_passengers = df.loc['1960', 'Passengers'].values

rmse = np.sqrt(mean_squared_error(actual_passengers, pred_passengers))
mape = np.mean(np.abs((actual_passengers - pred_passengers) / actual_passengers)) * 100
print(f"RMSE: {rmse:.2f}, MAPE: {mape:.2f}%")

In [ ]:
test_index = df.loc['1960'].index

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index, y=df['Passengers'],
    mode='lines', name='Historical (Train)',
    line=dict(color='steelblue')
))
fig.add_trace(go.Scatter(
    x=test_index, y=actual_passengers,
    mode='lines+markers', name='Actual (Test 1960)',
    line=dict(color='steelblue', dash='dot')
))
fig.add_trace(go.Scatter(
    x=test_index, y=pred_passengers,
    mode='lines+markers', name='AR(13) Forecast',
    line=dict(color='tomato')
))

fig.update_layout(
    title=f'Air Passengers — AR(13) Forecast vs Actual 1960  |  RMSE: {rmse:.1f}, MAPE: {mape:.1f}%',
    xaxis_title='Month', yaxis_title='Passengers',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99)
)
fig.show()

In [ ]:
residuals = actual_passengers - pred_passengers

fig_res = go.Figure()
fig_res.add_trace(go.Scatter(
    x=test_index, y=residuals,
    mode='lines+markers', name='Residuals',
    line=dict(color='seagreen')
))
fig_res.add_hline(y=0, line_dash='dash', line_color='red')
fig_res.update_layout(
    title='Forecast Residuals — 1960',
    xaxis_title='Month', yaxis_title='Residual (Actual − Predicted)',
    template='plotly_white'
)
fig_res.show()